# Retail Spending vs. Real GDP Downturns

Starter shared notebook for testing whether U.S. retail and food services spending declines in years when annual Real GDP growth is negative.

- Census source: Monthly Retail and Food Services Sales by Kind of Business, January 1992 through August 2026
- FRED source: Annual Real Gross Domestic Product (GDPCA), 1992 through 2025
- Shared overlap: 1992–2025; 2026 is retained only if a complete/usable Census period is available


In [ ]:
from pathlib import Path
import re

import pandas as pd

pd.options.display.float_format = "{:.2f}".format

DATA_DIR = Path("data")
CENSUS_PATH = DATA_DIR / "benchsales26.csv"
GDPCA_PATH = DATA_DIR / "GDPCA.csv"

print("Census file:", CENSUS_PATH)
print("GDPCA file:", GDPCA_PATH)


## Load and inspect both datasets

The Census file can be a standard rectangular CSV or a Census-style report with title/metadata rows. The helper below inspects both styles and is designed to accept a long-format file with `year`, `month`, and `retail_sales` columns or a wider file with month/year column labels.

In [ ]:
MONTH_MAP = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

def clean_sales(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text in {"", "(S)", "(NA)", "(*)"}:
        return None
    return pd.to_numeric(text.replace(",", ""), errors="coerce")

def load_census_retail(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Census retail file not found: {path}. Add the CSV under data/."
        )

    first = pd.read_csv(path)
    print("Census rectangular columns:", list(first.columns))
    print("Census rectangular shape:", first.shape)

    # Support a prepared rectangular/long format such as:
    # year, month, retail_sales
    lower_columns = {str(c).strip().lower(): c for c in first.columns}
    sales_column = next(
        (lower_columns[c] for c in ("retail_sales", "sales", "value") if c in lower_columns),
        None,
    )
    if sales_column and "year" in lower_columns:
        long = first.rename(columns={lower_columns["year"]: "year", sales_column: "retail_sales"}).copy()
        if "month" in lower_columns:
            long = long.rename(columns={lower_columns["month"]: "month"})
            long["month"] = pd.to_datetime(long["month"].astype(str), format="%b", errors="coerce").dt.month
        long["year"] = pd.to_numeric(long["year"], errors="coerce")
        long["retail_sales"] = pd.to_numeric(long["retail_sales"], errors="coerce")
        return long.dropna(subset=["year", "retail_sales"])

    # Census reports have metadata rows above a header row.
    raw = pd.read_csv(path, header=None, dtype=str, encoding="utf-8-sig")
    print("Census report shape:", raw.shape)

    kind_row = None
    for index, row in raw.iterrows():
        if row.astype(str).str.contains("Retail and food services sales, total", case=False, na=False).any():
            kind_row = index
            break
    if kind_row is None:
        raise ValueError("Could not find 'Retail and food services sales, total' in the Census file.")

    header_row = None
    year_by_column = {}
    for row_index in range(max(0, kind_row - 8), kind_row):
        month_columns = {}
        for column_index, value in enumerate(raw.iloc[row_index].tolist(), start=0):
            match = re.search(r"^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[.]?", str(value), re.I)
            if match:
                month_columns[column_index] = MONTH_MAP[match.group(1).lower()]
        if month_columns:
            header_row = row_index
            for column_index, month in month_columns.items():
                labeled_year = None
                for candidate_row in raw.iloc[max(0, row_index - 5):row_index + 1].values.tolist():
                    candidate_text = " ".join(str(v) for v in candidate_row if pd.notna(v))
                    years = re.findall(r"\b(?:19|20)\d{2}\b", candidate_text)
                    if years:
                        labeled_year = int(years[-1])
                        break
                explicit = re.search(r"\b(?:19|20)\d{2}\b", str(raw.iloc[row_index, column_index]))
                year_by_column[column_index] = int(explicit.group()) if explicit else labeled_year
            break

    if header_row is None:
        raise ValueError("Could not find month columns in the Census report.")

    records = []
    for column_index, month in month_columns.items():
        year = year_by_column.get(column_index)
        sales = clean_sales(raw.iloc[kind_row, column_index])
        if year is not None and sales is not None:
            records.append({"year": year, "month": month, "retail_sales": sales})

    census = pd.DataFrame(records)
    if census.empty:
        raise ValueError("No usable Census month values were parsed.")
    return census.sort_values(["year", "month"]).reset_index(drop=True)

census_monthly = load_census_retail(CENSUS_PATH)
print("\nCensus monthly preview:")
display(census_monthly.head())
print("Census year(s):", census_monthly["year"].unique().tolist())
print("Census monthly shape:", census_monthly.shape)


In [ ]:
if not GDPCA_PATH.exists():
    raise FileNotFoundError(
        f"GDPCA file not found: {GDPCA_PATH}. Download GDPCA.csv from FRED and add it under data/."
    )

gdp = pd.read_csv(GDPCA_PATH)
print("GDPCA columns:", list(gdp.columns))
print("GDPCA shape:", gdp.shape)
display(gdp.head())
display(gdp.tail())

if "observation_date" not in gdp.columns or "GDPCA" not in gdp.columns:
    raise ValueError("Expected GDPCA.csv columns: observation_date and GDPCA")

gdp["observation_date"] = pd.to_datetime(gdp["observation_date"])
gdp["year"] = gdp["observation_date"].dt.year
gdp = gdp[["year", "GDPCA"]].dropna().sort_values("year").reset_index(drop=True)

print("GDPCA year range:", int(gdp["year"].min()), "–", int(gdp["year"].max()))


## Annualize sales and calculate YoY change

This starter sums the available monthly retail observations for each year. For partial years, results are not directly comparable to full-year totals. The merge below restricts GDP to the data available in both sources.

In [ ]:
retail_annual = (
    census_monthly.groupby("year", as_index=False)
    .agg(
        retail_sales_annual=("retail_sales", "sum"),
        months_available=("month", "nunique"),
    )
    .sort_values("year")
)

retail_annual["retail_sales_yoy_pct"] = (
    retail_annual["retail_sales_annual"].pct_change(fill_method=None) * 100
)

gdp["GDPCA_yoy_pct"] = gdp["GDPCA"].pct_change(fill_method=None) * 100

print("Annual retail preview:")
display(retail_annual.head())
print("Annual GDPCA preview:")
display(gdp.head())


In [ ]:
merged = retail_annual.merge(gdp, on="year", how="inner")
merged = merged[(merged["year"] >= 1992) & (merged["year"] <= 2025)].sort_values("year").reset_index(drop=True)
merged["gdp_downturn"] = merged["GDPCA_yoy_pct"] < 0

print("Merged overlap shape:", merged.shape)
if merged.empty:
    print("No overlapping 1992–2025 retail observations. Add the full Census 1992–2025 file under data/.")
else:
    print("Merged overlap range:", int(merged["year"].min()), "–", int(merged["year"].max()))
display(merged)

downturn_years = merged.loc[merged["gdp_downturn"], ["year", "GDPCA_yoy_pct", "retail_sales_yoy_pct"]]
print("GDPCA downturn years:")
display(downturn_years)


## Visualization-ready data

`merged` contains one row per matched year and includes:

- `retail_sales_annual`: annual sum of available monthly Census sales
- `retail_sales_yoy_pct`: retail sales year-over-year percentage change
- `GDPCA`: annual Real GDP
- `GDPCA_yoy_pct`: Real GDP year-over-year percentage change
- `gdp_downturn`: `True` when Real GDP growth is negative

A complete Census 1992–2025 file is required for the analysis to have the intended historical observations.